# Model M1: Base Wavelet-PINNsFormer Architecture
Proposed Physics-Informed Transformer with Localized Wavelet Activation, EMA Dynamic NTK Weight Balancing, and Temporal Causality Mask for 4D Hodgkin-Huxley Dynamics.


In [ ]:
using Plots, Flux, NNlib, LinearAlgebra, Statistics, Random, CSV, DataFrames, Optim, Printf

include("architectures.jl")
include("loss_and_optimization.jl")


In [ ]:
# Load Hodgkin-Huxley Synthetic Ground-Truth Data
file_path = raw"C:\nirbhay\Downloads\NeuroPinnsFormmer-attention-that-neurons-needs\Synthetic_Data\HH_ground_truth_synthetic_data.csv"
HH_data = CSV.read(file_path, DataFrame)
first(HH_data, 5)


In [ ]:
# Instantiate Model M1 (Base Wavelet-PINNsFormer)
model_m1 = PINNsFormer(; d_model=32, k=10, n_heads=4, depth=1, act_type=:wavelet)
flat_params, _ = Flux.destructure(model_m1)
println("Total Model Parameters: ", length(flat_params))


In [ ]:
# Prepare Dataloader & Run Dual-Stage Optimization (30k Adam + 500 L-BFGS)
include("benchmark_runner.jl")
dataloader, _, _ = prepare_dataloader(HH_data, 10; batch_size=64)

trained_m1, wall_time = train_model_single_run(model_m1, dataloader; adam_epochs=1000, lbfgs_epochs=200, use_ntk=true)
metrics_m1 = evaluate_relative_l2_error(trained_m1, HH_data)

println("--> Training Completed in $(round(wall_time, digits=2)) seconds")
println("--> Final Relative L2 State Error: $(round(metrics_m1.l2_total, digits=6))")


In [ ]:
# Plot Membrane Voltage & Ion Channel Gating Dynamics
include("paper_plots.jl")
configure_paper_style()

t = HH_data.t
p_v = plot(t, HH_data.V, color=:black, line=:dash, label="Ground Truth (Radau)")
plot!(t, metrics_m1.V_pred, color=:crimson, label="Wavelet-PINNsFormer Prediction")
title!("Membrane Potential Dynamics V(t)")
ylabel!("Voltage (mV)")

p_g = plot(t, HH_data.m, color=:black, line=:dash, label="GT m")
plot!(t, metrics_m1.m_pred, color=:dodgerblue, label="Model m")
plot!(t, metrics_m1.h_pred, color=:darkorange, label="Model h")
plot!(t, metrics_m1.n_pred, color=:forestgreen, label="Model n")
title!("Ion Channel Gating Kinetics (m, h, n)")
xlabel!("Time (ms)")
ylabel!("Gating Probability")

dashboard = plot(p_v, p_g, layout=grid(2, 1, heights=[0.5, 0.5]), link=:x, size=(800, 600))
display(dashboard)
